# Modelo v2 — `demanda_lag_168`

Notebook propio de v2 (pliego `PLIEGO_Fase5bis_v2_lag168.md`, Parte 2 — PR A).
v2 = v1 + `demanda_lag_168`. Una sola feature nueva, nada más (pliego, 1.1).

Seis features: `hora`, `mes`, `tipo_efectivo`, `es_puente`, `demanda_lag_24`,
`demanda_lag_168`. Hiperparámetros heredados de v1 sin barrido nuevo
(`max_depth=10`, `min_samples_leaf=5`).

**Cero lecturas del conjunto de test (2026) en este notebook.** Solo se
materializan filas de 2023, 2024 y 2025; la variable `test`/`test_model` de
`notebooks/modelo_demanda.ipynb` no tiene equivalente aquí a propósito.

In [1]:
import sys
from pathlib import Path

for candidato in (Path.cwd(), *Path.cwd().resolve().parents):
    if (candidato / "src").is_dir():
        if str(candidato) not in sys.path:
            sys.path.insert(0, str(candidato))
        break

from src.paths import RAIZ, DIR_PROCESSED

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

from src.features import FEATURES_5F
from src.features_v2 import FEATURES_6F, VARIABLE_OBJETIVO, construir_features_6f
from src.modeling import barrido_validacion, dispersion_top

## 1. Cargar histórico y construir las 6 features

In [3]:
df = pd.read_parquet(DIR_PROCESSED / "demanda_horaria.parquet")
df_features = construir_features_6f(df)
print(f"Histórico completo: {len(df_features)} filas, "
      f"{df_features['datetime_utc'].min()} -> {df_features['datetime_utc'].max()}")

Histórico completo: 30648 filas, 2023-01-01 00:00:00+00:00 -> 2026-06-30 23:00:00+00:00


## 2. Partición train (2023-2025) y prefijo de NaN (168h)

Test (2026) nunca se materializa: solo se filtran los años 2023-2025 desde
`df_features`. El horizonte de arranque cambia respecto a v1 -- `demanda_lag_168`
deja NaN en las primeras 168 horas de la serie, no en las primeras 24
(pliego, 1.2) -- así que el prefijo descartado crece de 24 a 168 horas.

In [4]:
años_utc = df_features["datetime_utc"].dt.year
train_raw_6f = df_features[años_utc.isin([2023, 2024, 2025])].copy()

n_antes = len(train_raw_6f)
train_clean_6f = train_raw_6f.dropna(subset=["demanda_lag_24", "demanda_lag_168"]).copy()
n_despues = len(train_clean_6f)
n_descartadas = n_antes - n_despues

print(f"Filas 2023-2025 antes del dropna del prefijo: {n_antes}")
print(f"Filas descartadas por el prefijo de NaN (168h): {n_descartadas}")
print(f"Filas de train tras el prefijo: {n_despues}")
print(f"Rango: {train_clean_6f['datetime_utc'].min()} -> {train_clean_6f['datetime_utc'].max()}")

assert n_descartadas == 168, f"Se esperaban 168 horas descartadas por el prefijo, hay {n_descartadas}"
assert n_despues == 26136, f"Se esperaban 26.136 filas de train, hay {n_despues}"


Filas 2023-2025 antes del dropna del prefijo: 26304
Filas descartadas por el prefijo de NaN (168h): 168
Filas de train tras el prefijo: 26136
Rango: 2023-01-08 00:00:00+00:00 -> 2025-12-31 23:00:00+00:00


## 3. v1-comparable vs v2 en VALIDACIÓN 2025

Mismo split que v1: entrenar 2023-2024 (`train_fit_6f`), elegir en 2025
(`val_6f`). Mismos hiperparámetros heredados para ambas columnas de
features -- la única diferencia es `demanda_lag_168`. Referencia documental
(pliego, 1.3): el despliegue no se condiciona a que v2 mejore aquí.

In [5]:
PARAMS_HEREDADOS = {"max_depth": 10, "min_samples_leaf": 5}
obj_nivel = lambda d: d[VARIABLE_OBJETIVO]

años_sel = train_clean_6f["datetime_utc"].dt.year
train_fit_6f = train_clean_6f[años_sel <= 2024].copy()
val_6f = train_clean_6f[años_sel == 2025].copy()

assert len(train_fit_6f) + len(val_6f) == len(train_clean_6f), "el corte pierde filas"
assert train_fit_6f["datetime_utc"].max() < val_6f["datetime_utc"].min(), "solapamiento fit/val"
print(f"train_fit_6f {len(train_fit_6f)} | val_6f {len(val_6f)}")

def _mae_sobre_val(features):
    modelo = DecisionTreeRegressor(random_state=42, **PARAMS_HEREDADOS)
    modelo.fit(train_fit_6f[features], obj_nivel(train_fit_6f))
    pred = modelo.predict(val_6f[features])
    y = obj_nivel(val_6f)
    return mean_absolute_error(y, pred), (y - pred).mean()

MAE_VAL_V1C, SESGO_VAL_V1C = _mae_sobre_val(FEATURES_5F)   # v1-comparable
MAE_VAL_V2, SESGO_VAL_V2 = _mae_sobre_val(FEATURES_6F)     # v2
DELTA_VAL = MAE_VAL_V2 - MAE_VAL_V1C

print()
print(f"v1-comparable (5f, heredados) MAE val: {MAE_VAL_V1C:.2f} MW | sesgo {SESGO_VAL_V1C:+.2f} MW")
print(f"v2 (6f, heredados)            MAE val: {MAE_VAL_V2:.2f} MW | sesgo {SESGO_VAL_V2:+.2f} MW")
print(f"Diferencia (v2 - v1)                 : {DELTA_VAL:+.2f} MW")

train_fit_6f 17376 | val_6f 8760

v1-comparable (5f, heredados) MAE val: 987.15 MW | sesgo +215.96 MW
v2 (6f, heredados)            MAE val: 997.25 MW | sesgo +124.50 MW
Diferencia (v2 - v1)                 : +10.10 MW


## 4. Rejilla sobre 6f -- dispersión del top-5

Misma rejilla que produjo v1 (`notebooks/modelo_demanda.ipynb`, celda 21,
líneas 82-83), copiada literal. No se usa para elegir hiperparámetros --
v2 hereda los de v1 sin barrido nuevo (pliego) -- solo para medir la
dispersión del top-5 como referencia documental.

In [6]:
rejilla_depth = [4, 6, 8, 10, 12, 15, 20, None]
rejilla_leaf = [1, 5, 20, 50]

ranking_6f = barrido_validacion(FEATURES_6F, obj_nivel, train_fit_6f, val_6f,
                                rejilla_depth, rejilla_leaf)
DISP_TOP5_6F = dispersion_top(ranking_6f)

print(ranking_6f[["max_depth", "min_samples_leaf", "mae_fit", "mae_val", "gap", "sesgo_val"]]
      .head(5).to_string(index=False))
print()
print(f"Dispersión del top-5: {DISP_TOP5_6F:.2f} MW")

max_depth  min_samples_leaf    mae_fit     mae_val        gap  sesgo_val
       10                 1 628.026806  996.910274 368.883468 121.819275
       10                 5 651.599572  997.253394 345.653821 124.504267
       10                20 707.434709  999.851149 292.416439 125.974091
       12                20 678.927891 1003.950847 325.022956 128.520186
       20                20 672.093495 1008.691922 336.598427 129.391417

Dispersión del top-5: 11.78 MW


## 5. Determinismo -- dos entrenamientos seguidos dan el mismo modelo

In [7]:
modelo_a = DecisionTreeRegressor(max_depth=10, min_samples_leaf=5, random_state=42)
modelo_a.fit(train_clean_6f[FEATURES_6F], obj_nivel(train_clean_6f))

modelo_b = DecisionTreeRegressor(max_depth=10, min_samples_leaf=5, random_state=42)
modelo_b.fit(train_clean_6f[FEATURES_6F], obj_nivel(train_clean_6f))

pred_a = modelo_a.predict(train_clean_6f[FEATURES_6F])
pred_b = modelo_b.predict(train_clean_6f[FEATURES_6F])

assert np.array_equal(pred_a, pred_b), "Dos entrenamientos seguidos no dan el mismo modelo"
assert np.array_equal(modelo_a.feature_importances_, modelo_b.feature_importances_)
print("Determinismo confirmado: dos entrenamientos seguidos con random_state=42 "
      "dan predicciones e importancias de features identicas.")

Determinismo confirmado: dos entrenamientos seguidos con random_state=42 dan predicciones e importancias de features identicas.


## 6. Modelo final de v2 (el que se serializa)

Entrenado sobre `train_clean_6f` completo (2023-01-08 -> 2025-12-31,
26.136 filas), mismo patrón que v1 (celda 28 de `modelo_demanda.ipynb`):
el ganador se reentrena con todo el histórico de train disponible.

In [8]:
params_v2 = {"max_depth": 10, "min_samples_leaf": 5}
modelo_lag168_v2 = DecisionTreeRegressor(random_state=42, **params_v2)
modelo_lag168_v2.fit(train_clean_6f[FEATURES_6F], obj_nivel(train_clean_6f))

print(f"modelo_lag168_v2 entrenado sobre train_clean_6f completo "
      f"({len(train_clean_6f)} filas, "
      f"{train_clean_6f['datetime_utc'].min()} -> {train_clean_6f['datetime_utc'].max()}).")

modelo_lag168_v2 entrenado sobre train_clean_6f completo (26136 filas, 2023-01-08 00:00:00+00:00 -> 2025-12-31 23:00:00+00:00).
